# 14c — Pull RSUI (Reported Social Unrest Index)

**Source:** Barrett, Appendino, Nguyen, De la Torre (2021). *"Toward a Better Measurement of Social Unrest."*
IMF Working Paper WP/21/291.  
**Coverage:** ~130 countries, 1996–present, monthly  
**Access:** Direct CSV download from IMF website  

## What this notebook does

1. Downloads the RSUI monthly panel from the IMF data release
2. Aggregates to country-year (mean, max, and threshold-exceedance flag)
3. Derives the standard binary label used in the literature:
   `unrest_binary` = 1 if the annual max RSUI score exceeds the country-specific
   75th-percentile threshold (the operationalisation in Barrett et al. 2021)
4. Writes both the monthly and annual panels to ADLS

## Why this matters

RSUI is the **primary dependent variable** in the IMF WP/21/291 social unrest model,
which is the most-cited study in the synthesis document for mass-protest prediction.
It is constructed from Factiva newspaper article counts (mentions of protest/riot/unrest
within 10 words of a country name) and is independent of ACLED or GDELT event coding,
making it a complementary and cross-validating label source.

The synthesis document identifies three sub-type labels derived from RSUI:
- `unrest_type_gov` — government-related unrest
- `unrest_type_election` — election-related unrest  
- `unrest_type_violent` — violent unrest (hardest to predict)

These sub-types are included in the IMF data release and are written here.

## Output paths on ADLS

```
raw/rsui/{RUN_DATE}/rsui_monthly.parquet
raw/rsui/{RUN_DATE}/rsui_annual.parquet
```

Notebook 14 should add `"rsui": "raw/rsui"` to its `RAW_PREFIXES` dict.

In [ ]:
%%capture
%pip install requests pandas pyarrow azure-identity adlfs openpyxl --quiet

In [ ]:
import io
import os
import warnings
from datetime import datetime
from pathlib import Path

import pandas as pd
import numpy as np
import requests
import adlfs
from azure.identity import DefaultAzureCredential

warnings.filterwarnings('ignore')

RUN_DATE = datetime.utcnow().strftime('%Y%m%d')

In [ ]:
ADLS_ACCOUNT_NAME = os.environ['ADLS_ACCOUNT_NAME']
ADLS_CONTAINER    = os.getenv('ADLS_CONTAINER', 'data')

# IMF RSUI data release.
# Primary URL: IMF WP/21/291 supplementary data (direct Excel download).
# If the primary URL changes, check:
#   https://www.imf.org/en/Publications/WP/Issues/2021/11/05/ (search WP/21/291)
# A mirrored CSV is also available on the Harvard Dataverse (see fallback below).
RSUI_PRIMARY_URL = (
    'https://www.imf.org/-/media/Files/Publications/WP/2021/'
    'English/wpiea2021291-print-pdf.ashx'
)
# The actual data file (Excel) linked from the working paper abstract page:
RSUI_DATA_URL = (
    'https://www.imf.org/-/media/Files/Publications/WP/2021/'
    'Datasets/wpiea2021291-dataset.ashx'
)

# Expected column names in the IMF release (may vary slightly by release version)
# The dataset is a long panel: country, year, month, rsui, and sub-type scores
RSUI_COUNTRY_COL = 'iso3'       # will be mapped/renamed to this
RSUI_YEAR_COL    = 'year'
RSUI_MONTH_COL   = 'month'
RSUI_SCORE_COL   = 'rsui'

# Annual threshold for binary label: country's own historical Nth percentile
RSUI_THRESHOLD_PERCENTILE = 75

In [ ]:
credential = DefaultAzureCredential()
storage_options = {'account_name': ADLS_ACCOUNT_NAME, 'credential': credential}


def adls_path(subpath):
    return (f'abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}'
            f'.dfs.core.windows.net/{subpath}')


def write_parquet(df, subpath):
    df.to_parquet(adls_path(subpath), storage_options=storage_options,
                  index=False, engine='pyarrow')
    print(f'  Written {len(df):,} rows → {subpath}')

In [ ]:
def _download_rsui(url, timeout=60):
    """Download the IMF RSUI Excel file and return a DataFrame."""
    print(f'Downloading RSUI from {url} ...')
    resp = requests.get(url, timeout=timeout,
                        headers={'User-Agent': 'Mozilla/5.0'})
    resp.raise_for_status()
    # The IMF release is an Excel workbook; the data sheet is typically named
    # 'Data' or 'RSUI'. Try both.
    xl = pd.ExcelFile(io.BytesIO(resp.content))
    print(f'  Sheets available: {xl.sheet_names}')
    data_sheet = next(
        (s for s in xl.sheet_names if 'data' in s.lower() or 'rsui' in s.lower()),
        xl.sheet_names[0],
    )
    print(f'  Reading sheet: {data_sheet}')
    return pd.read_excel(xl, sheet_name=data_sheet)


def _normalise_rsui(df_raw):
    """
    Normalise the raw RSUI download into a tidy long panel with columns:
      iso3, year, month, rsui, rsui_gov, rsui_election, rsui_violent

    The IMF release has varied slightly across versions; this function
    handles both the wide (country × date columns) and long formats.
    """
    df = df_raw.copy()
    df.columns = [str(c).strip().lower() for c in df.columns]

    # ── Long format (already tidy) ─────────────────────────────────────────
    if 'iso3' in df.columns or 'country_code' in df.columns:
        df = df.rename(columns={'country_code': 'iso3', 'iso_code': 'iso3',
                                 'iso': 'iso3', 'rsui_index': 'rsui',
                                 'rsui_score': 'rsui'})
        for sub in ('gov', 'election', 'violent'):
            # Accept various naming conventions
            candidates = [c for c in df.columns if sub in c and 'rsui' in c]
            if candidates:
                df = df.rename(columns={candidates[0]: f'rsui_{sub}'})
            else:
                df[f'rsui_{sub}'] = np.nan
        keep = ['iso3', 'year', 'month', 'rsui', 'rsui_gov', 'rsui_election', 'rsui_violent']
        return df[[c for c in keep if c in df.columns]].copy()

    # ── Wide format (countries as rows, date columns like '1996m1') ────────
    # First column is country identifier; remaining are date-labelled values
    id_cols = [c for c in df.columns if not c[0].isdigit()]
    val_cols = [c for c in df.columns if c not in id_cols]
    df_long = df.melt(id_vars=id_cols, value_vars=val_cols,
                      var_name='date_str', value_name='rsui')
    # Parse '1996m1' → year=1996, month=1
    df_long[['year', 'month']] = (
        df_long['date_str'].str.extract(r'(\d{4})m(\d{1,2})')
        .astype(float)
    )
    # Identify country column
    iso_col = next((c for c in id_cols if 'iso' in c or 'code' in c or 'country' in c), id_cols[0])
    df_long = df_long.rename(columns={iso_col: 'iso3'})
    for sub in ('gov', 'election', 'violent'):
        df_long[f'rsui_{sub}'] = np.nan
    keep = ['iso3', 'year', 'month', 'rsui', 'rsui_gov', 'rsui_election', 'rsui_violent']
    return df_long[[c for c in keep if c in df_long.columns]].dropna(subset=['year', 'month'])


# Download with fallback
df_raw = None
for url in [RSUI_DATA_URL, RSUI_PRIMARY_URL]:
    try:
        df_raw = _download_rsui(url)
        print(f'  Raw shape: {df_raw.shape}')
        break
    except Exception as exc:
        print(f'  Failed ({url}): {exc}')

if df_raw is None:
    raise RuntimeError(
        'Could not download RSUI data from any URL. '
        'Download the Excel file manually from the IMF WP/21/291 abstract page '
        'and load it with: df_raw = pd.read_excel(\'path/to/file.xlsx\')'
    )

In [ ]:
df_monthly = _normalise_rsui(df_raw)
df_monthly['year']  = df_monthly['year'].astype(int)
df_monthly['month'] = df_monthly['month'].astype(int)
df_monthly = df_monthly.sort_values(['iso3', 'year', 'month']).reset_index(drop=True)

print(f'Monthly panel : {len(df_monthly):,} rows')
print(f'Countries     : {df_monthly["iso3"].nunique()}')
print(f'Date range    : {df_monthly["year"].min()}m{df_monthly.loc[df_monthly["year"]==df_monthly["year"].min(),"month"].min()} '
      f'– {df_monthly["year"].max()}m{df_monthly.loc[df_monthly["year"]==df_monthly["year"].max(),"month"].max()}')
df_monthly.head()

In [ ]:
# Aggregate to country-year and derive the standard binary label

def _annual_label(group, score_col, pct):
    """1 if any month in the year exceeds this country's historical Nth percentile."""
    threshold = group[score_col].quantile(pct / 100)
    return (group[score_col].max() > threshold).astype(int)


annual_rows = []
for (iso3, year), grp in df_monthly.groupby(['iso3', 'year']):
    row = {
        'iso3':             iso3,
        'year':             year,
        'rsui_mean':        grp['rsui'].mean(),
        'rsui_max':         grp['rsui'].max(),
        'rsui_std':         grp['rsui'].std(),
        # Country-specific threshold binary label (Barrett et al. 2021)
        'unrest_binary':    int(grp['rsui'].max() > 0),  # placeholder; recalculated below
    }
    for sub in ('gov', 'election', 'violent'):
        col = f'rsui_{sub}'
        if col in grp.columns:
            row[f'rsui_{sub}_mean'] = grp[col].mean()
            row[f'rsui_{sub}_max']  = grp[col].max()
    annual_rows.append(row)

df_annual = pd.DataFrame(annual_rows)

# Apply the country-specific percentile threshold across all years for each country
threshold_map = (
    df_monthly.groupby('iso3')['rsui']
    .quantile(RSUI_THRESHOLD_PERCENTILE / 100)
    .rename('country_threshold')
)
df_annual = df_annual.merge(threshold_map, on='iso3', how='left')
df_annual['unrest_binary'] = (df_annual['rsui_max'] > df_annual['country_threshold']).astype(int)
df_annual = df_annual.drop(columns='country_threshold')

df_annual = df_annual.sort_values(['iso3', 'year']).reset_index(drop=True)

print(f'Annual panel  : {len(df_annual):,} country-years')
print(f'unrest_binary base rate: {df_annual["unrest_binary"].mean():.1%}')
df_annual.head(10)

In [ ]:
write_parquet(df_monthly, f'raw/rsui/{RUN_DATE}/rsui_monthly.parquet')
write_parquet(df_annual,  f'raw/rsui/{RUN_DATE}/rsui_annual.parquet')

print()
print('=' * 55)
print('Notebook 14c complete')
print('=' * 55)
print(f'  Monthly rows  : {len(df_monthly):,}')
print(f'  Annual rows   : {len(df_annual):,}')
print(f'  Countries     : {df_annual["iso3"].nunique()}')
print()
print('Labels written:')
print('  unrest_binary       — 1 if annual max RSUI > country 75th-percentile')
print('  rsui_mean/max/std   — continuous aggregates for use as predictors')
print()
print('Next: add  "rsui": "raw/rsui"  to RAW_PREFIXES in notebook 14.')